In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from sklearn.metrics import r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.utils import plot_model

pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
# Initialize random seeds for reproducibility

dictionary = pd.read_csv('FINANCE384_stock_month_data_dictionary.csv')
display(dictionary)

/Users/thismac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,variable,description,role
0,date,End of month t (CRSP month-end trading date),identifier; not a predictor
1,permno,CRSP permanent security identifier,identifier; not a predictor
2,ticker,Exchange ticker at month t,identifier; not a predictor
3,comnam,Company name at month t,identifier; not a predictor
4,industry,Coarse SIC division label (Agriculture collaps...,"classification, known at t; permitted as a pre..."
5,ff49_code,Fama-French 49-industry code (Ships collapsed ...,"classification, known at t; permitted as a pre..."
6,ff49_name,Fama-French 49-industry name (Ships collapsed ...,"classification, known at t; permitted as a pre..."
7,gvkey,Compustat firm identifier (missing if no CCM l...,identifier; not a predictor
8,size,log market capitalization (USD) at end of mont...,"characteristic, contemporaneous with t"
9,bm,Book equity (6-month availability lag) / marke...,"characteristic, contemporaneous with t"


In [3]:
determined_characteristics = dictionary.loc[
    dictionary["role"] == "characteristic, contemporaneous with t", "variable"].tolist()

predetermined_characteristics = dictionary.loc[
    dictionary["role"] == "characteristic, predetermined at t-1", "variable"].tolist()

determined_classifications = dictionary.loc[
    dictionary["role"] == "classification, known at t; permitted as a predictor if encoded", "variable"].tolist()

print(f"Rows: {len(dictionary):,}")
print(f"Determined characteristics: {determined_characteristics}")
print(f"Predetermined characteristics: {predetermined_characteristics}")
print(f"Determined classifications: {determined_classifications}")

FEATURES = determined_characteristics + predetermined_characteristics
print("Number of features:", len(FEATURES))

Rows: 27
Determined characteristics: ['size', 'bm', 'vol12', 'beta60', 'ivol60', 'turnover', 'dollar_volume', 'amihud_illiq', 'divyield', 'mkt_12m', 'mkt_vol_12m', 'down_market']
Predetermined characteristics: ['mom12_2', 'gross_profit', 'roe', 'asset_growth', 'leverage', 'accruals']
Determined classifications: ['industry', 'ff49_code', 'ff49_name']
Number of features: 18


In [4]:
panel = pd.read_csv('FINANCE384_assignmentA_development_panel.csv', parse_dates=['date'])

print(f"Rows           : {len(panel):,}")
print(f"Distinct stocks: {panel['permno'].nunique():,}")
print(f"Months         : {panel['date'].nunique():,} "
      f"({panel['date'].min():%Y-%m} to {panel['date'].max():%Y-%m})")
panel.head()


print(f"Duplicate (permno, date) rows: {panel.duplicated(['permno', 'date']).sum()}")

# Check 2: how many stocks are in each month, and how many months does a stock survive?
stocks_per_month = panel.groupby("date")["permno"].nunique()
months_per_stock = panel.groupby("permno")["date"].nunique()
print(f"Stocks per month : {stocks_per_month.min()} to {stocks_per_month.max()} "
      f"(mean {stocks_per_month.mean():.0f})")
print(f"Months per stock : {months_per_stock.min()} to {months_per_stock.max()} "
      f"(median {months_per_stock.median():.0f})")


Rows           : 198,298
Distinct stocks: 1,252
Months         : 396 (1990-01 to 2022-12)
Duplicate (permno, date) rows: 0
Stocks per month : 497 to 505 (mean 501)
Months per stock : 1 to 396 (median 122)


In [5]:
TARGET = "target_ret_excess_tp1"

panel["month"] = panel["date"].dt.to_period("M")

next_month_return = (
    panel[["permno", "month", "ret_excess_t"]]
    .rename(columns={"ret_excess_t": TARGET})
    .assign(month=lambda df: df["month"] - 1)
)

panel = panel.merge(next_month_return, on=["permno", "month"], how="left", validate="one_to_one")

final_month = (
    panel.groupby("permno")["month"]
    .transform("max")
    .eq(panel["month"])
)

no_target = panel[TARGET].isna()
# Diagnostics
print(f"Rows: {len(panel):,}")
print(f"Usable targets: {panel[TARGET].notna().sum():,}")
print(f"No month t+1 in data: {no_target.sum():,}")
print(f"Stock's last month: {(no_target & final_month).sum():,}")
print(f"Gap in coverage: {(no_target & ~final_month).sum():,}")

print("\nMissing values:")
print(panel[FEATURES + [TARGET]].isna().sum())

print("\nFeature summary:")
display(
    panel[FEATURES + [TARGET]]
    .describe()
    .loc[["mean", "std", "min", "max"]]
)

panel[["permno", "month", "ret_excess_t", TARGET]].head()

Rows: 198,298
Usable targets: 197,016
No month t+1 in data: 1,282
Stock's last month: 1,252
Gap in coverage: 30

Missing values:
size                         0
bm                        4242
vol12                     5268
beta60                   17211
ivol60                   17211
turnover                     0
dollar_volume                0
amihud_illiq                 0
divyield                  5927
mkt_12m                      0
mkt_vol_12m                  0
down_market                  0
mom12_2                   6023
gross_profit               515
roe                       4259
asset_growth               737
leverage                   467
accruals                  6615
target_ret_excess_tp1     1282
dtype: int64

Feature summary:


,size,bm,vol12,beta60,ivol60,turnover,dollar_volume,amihud_illiq,divyield,mkt_12m,mkt_vol_12m,down_market,mom12_2,gross_profit,roe,asset_growth,leverage,accruals,target_ret_excess_tp1
mean,23.0467,0.5098,0.0847,1.0445,0.0725,0.1647,3.3957e+09,0.0003,0.0207,0.0891,0.0404,0.2297,0.1158,0.2994,0.1733,0.1099,0.2625,-0.0482,0.0083
std,1.3306,0.4449,0.0452,0.5579,0.0304,0.1518,6.9844e+09,0.0018,0.0196,0.1661,0.0164,0.4207,0.3274,0.2254,0.2518,0.2472,0.1713,0.0555,0.0990
min,17.3590,0.0082,0.0193,-0.3211,0.0257,0.0104,9.6556e+05,0.0000,0.0000,-0.4330,0.0108,0.0000,-0.9108,-0.1985,-0.8534,-0.4729,0.0000,-0.6748,-0.8703
max,27.5092,11.2483,0.5192,4.0973,0.2498,2.1366,2.3617e+11,0.1307,0.5481,0.6466,0.0894,1.0000,4.2891,1.1964,4.5006,3.0477,1.0184,0.1612,2.6266


,permno,month,ret_excess_t,target_ret_excess_tp1
0,10057,1990-01,-0.0540,-0.0179
1,10057,1990-02,-0.0179,-0.0558
2,10057,1990-03,-0.0558,-0.0485
3,10057,1990-04,-0.0485,-0.0205
4,10057,1990-05,-0.0205,-0.1035


In [6]:
industry = 'ff49_name'

for feature in FEATURES:
    monthly_median = (
        panel.groupby('date')[feature]
        .transform('median'))
    
    panel[feature] = (
        panel[feature]
        .fillna(monthly_median))

panel = panel.loc[panel[TARGET].notna()].copy()

print("\nMissing values:")
print(panel[FEATURES + [TARGET]].isna().sum())


Missing values:
size                     0
bm                       0
vol12                    0
beta60                   0
ivol60                   0
turnover                 0
dollar_volume            0
amihud_illiq             0
divyield                 0
mkt_12m                  0
mkt_vol_12m              0
down_market              0
mom12_2                  0
gross_profit             0
roe                      0
asset_growth             0
leverage                 0
accruals                 0
target_ret_excess_tp1    0
dtype: int64


In [7]:
train = panel.loc[panel['month'].between('1990-01', '2014-12')].copy()
valid = panel.loc[panel['month'].between('2015-01', '2018-12')].copy()
test = panel.loc[panel['month'].between('2019-01', '2022-11')].copy()
